# NanoChemGPT: Complete Usage Examples

This notebook demonstrates the key functionality of NanoChemGPT, a domain-specific RAG system for nanochemistry literature mining and synthesis planning.

## Table of Contents
1. [Setup and Configuration](#setup)
2. [Basic Question Answering](#basic-qa)
3. [File Upload and Analysis](#file-upload)
4. [Protocol Conversion](#protocol-conversion)
5. [Literature Mining](#literature-mining)
6. [Citation Management](#citation-management)
7. [System Evaluation](#evaluation)
8. [Advanced Features](#advanced)

## Prerequisites
- NanoChemGPT server running locally or remotely
- OpenAI API key configured
- Required Python packages: `requests`, `pandas`, `matplotlib`

## 1. Setup and Configuration {#setup}

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time
from typing import Dict

# Configuration
BASE_URL = "http://localhost:5000"  # Change to your server URL
HEADERS = {"Content-Type": "application/json"}


def make_request(
    endpoint: str, method: str = "GET", data: Dict = None, files: Dict = None
) -> Dict:
    """Helper function to make API requests."""
    url = f"{BASE_URL}{endpoint}"
    try:
        if method == "POST":
            if files:
                response = requests.post(url, data=data, files=files)
            else:
                response = requests.post(url, json=data, headers=HEADERS)
        else:
            response = requests.get(url, params=data)

        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return {"ok": False, "error": str(e)}


# Test server connection
health_check = make_request("/health")
print(f"Server status: {health_check}")

## 2. Basic Question Answering {#basic-qa}

Let's start with basic nanochemistry questions to demonstrate the system's capabilities.

In [ ]:
# Example 1: Gold nanoparticle synthesis
query_1 = {
    "question": "How to synthesize gold nanoparticles using citrate reduction?",
    "mode": "protocol",
    "k_doc": 5,
    "k_passage": 10,
}

response_1 = make_request("/ask", "POST", query_1)

if response_1.get("ok"):
    print("🧪 Gold Nanoparticle Synthesis Protocol:")
    print("=" * 50)
    print(response_1["answer"])
    print("\n📚 References:")
    for i, ref in enumerate(response_1.get("refs", []), 1):
        print(f"[{i}] {ref.get('title', 'Unknown')} - {ref.get('authors', 'Unknown')}")
else:
    print(f"Error: {response_1.get('error')}")

In [ ]:
# Example 2: Silver nanoparticle synthesis
query_2 = {
    "question": "What are the optimal conditions for silver nanoparticle synthesis using chemical reduction?",
    "mode": "reasoning",
    "intent": "analysis",
}

response_2 = make_request("/ask", "POST", query_2)

if response_2.get("ok"):
    print("🔬 Silver Nanoparticle Analysis:")
    print("=" * 50)
    print(response_2["answer"])

    if "rationale" in response_2:
        print("\n🧠 Scientific Rationale:")
        print(response_2["rationale"])
else:
    print(f"Error: {response_2.get('error')}")

## 3. File Upload and Analysis {#file-upload}

Demonstrate how to upload synthesis protocols and analyze them.

In [ ]:
# Create a sample synthesis protocol file
sample_protocol = """
Gold Nanoparticle Synthesis Protocol

Materials:
- HAuCl₄·3H₂O (0.5 mM, 100 mL)
- Sodium citrate (38.8 mM, 10 mL)
- Deionized water

Procedure:
1. Heat 100 mL of 0.5 mM HAuCl₄ solution to boiling (100°C)
2. Add 10 mL of 38.8 mM sodium citrate solution rapidly
3. Continue boiling for 15 minutes while stirring
4. Cool to room temperature
5. Centrifuge at 8000 rpm for 10 minutes
6. Wash with deionized water twice

Expected Results:
- Particle size: 15-20 nm
- Color: Deep red
- Yield: ~85%
"""

# Save to temporary file
protocol_file = Path("sample_protocol.txt")
protocol_file.write_text(sample_protocol)

print(f"Created sample protocol file: {protocol_file}")
print(f"File size: {protocol_file.stat().st_size} bytes")

In [ ]:
# Upload and analyze the protocol
with open(protocol_file, "rb") as f:
    files = {"file": ("sample_protocol.txt", f, "text/plain")}
    data = {
        "question": "Analyze this synthesis protocol and suggest optimizations for better yield and particle size control",
        "mode": "reasoning",
    }

    response_3 = make_request("/ask", "POST", data, files)

if response_3.get("ok"):
    print("📄 Protocol Analysis Results:")
    print("=" * 50)
    print(response_3["answer"])

    # Check context statistics
    if "context_stats" in response_3:
        stats = response_3["context_stats"]
        print("\n📊 Context Statistics:")
        for key, value in stats.items():
            print(f"  {key}: {value}")
else:
    print(f"Error: {response_3.get('error')}")

# Clean up
protocol_file.unlink()

## 4. Protocol Conversion {#protocol-conversion}

Convert free-text protocols to structured robot operations.

In [ ]:
# Convert protocol text to robot operations
protocol_text = """
Heat the gold chloride solution to 100°C while stirring at 300 rpm. 
Add 10 mL of sodium citrate dropwise over 2 minutes. 
Continue heating for 15 minutes. 
Cool to room temperature and centrifuge at 8000 rpm for 10 minutes.
"""

conversion_request = {
    "text": protocol_text,
    "target_ops": ["heat", "mix", "add", "wait", "cool", "centrifuge"],
    "validate": True,
}

response_4 = make_request("/convert", "POST", conversion_request)

if response_4.get("ok"):
    print("🤖 Robot Operations:")
    print("=" * 50)

    operations = response_4["operations"]
    for i, op in enumerate(operations, 1):
        print(f"Step {i}: {op['action']}")
        for key, value in op.items():
            if key != "action":
                print(f"  {key}: {value}")
        print()

    # Validation results
    if "validation" in response_4:
        validation = response_4["validation"]
        print(f"✅ Validation: {'Passed' if validation['valid'] else 'Failed'}")
        if validation.get("warnings"):
            print(f"⚠️  Warnings: {validation['warnings']}")
        if validation.get("missing_ops"):
            print(f"❌ Missing operations: {validation['missing_ops']}")
else:
    print(f"Error: {response_4.get('error')}")

## 5. Literature Mining {#literature-mining}

Trigger background literature mining for specific topics.

In [ ]:
# Start a literature mining job
mining_request = {
    "query": "quantum dots synthesis CdSe semiconductor nanocrystals",
    "max_results": 20,
    "sources": ["eupmc", "arxiv"],
    "min_year": 2020,
    "filters": {"open_access": True, "peer_reviewed": True},
}

response_5 = make_request("/mine", "POST", mining_request)

if response_5.get("ok"):
    job_id = response_5["job_id"]
    print(f"📚 Literature mining started: {job_id}")
    print(f"Query: {response_5['query']}")
    print(f"Estimated time: {response_5['estimated_time']}")

    # Monitor job progress (simplified for demo)
    for attempt in range(3):
        time.sleep(10)  # Wait 10 seconds
        status_response = make_request(f"/mine/{job_id}")

        if status_response.get("ok"):
            status = status_response["status"]
            progress = status_response.get("progress", 0)
            print(f"Status: {status} ({progress}%)")

            if status == "completed":
                results = status_response["results"]
                print("\n📊 Mining Results:")
                print(f"  Papers found: {results['papers_found']}")
                print(f"  Papers processed: {results['papers_processed']}")
                print(f"  Entities extracted: {results['entities_extracted']}")
                print(f"  Index updated: {results['index_updated']}")
                break
            elif status == "error":
                print(f"❌ Mining failed: {status_response.get('error')}")
                break
        else:
            print(f"Failed to check status: {status_response.get('error')}")
            break
else:
    print(f"Error starting mining: {response_5.get('error')}")

## 6. Citation Management {#citation-management}

Demonstrate the citation tracking and reference formatting capabilities.

In [ ]:
# Query with explicit citation requirements
citation_query = {
    "question": "What is the Turkevich method for gold nanoparticle synthesis? Include specific reaction conditions and cite all sources.",
    "mode": "protocol",
    "want_inline": True,
    "k_doc": 8,
}

response_6 = make_request("/ask", "POST", citation_query)

if response_6.get("ok"):
    answer = response_6["answer"]
    refs = response_6.get("refs", [])

    print("📝 Response with Citations:")
    print("=" * 50)
    print(answer)

    print("\n📚 Reference List:")
    print("=" * 50)
    for i, ref in enumerate(refs, 1):
        citation = f"[{i}] "
        if ref.get("authors"):
            citation += f"{ref['authors']}. "
        if ref.get("title"):
            citation += f"\"{ref['title']}\". "
        if ref.get("journal"):
            citation += f"{ref['journal']}. "
        if ref.get("year"):
            citation += f"({ref['year']}). "
        if ref.get("doi"):
            citation += f"DOI: {ref['doi']}"
        elif ref.get("url"):
            citation += f"URL: {ref['url']}"

        print(citation)

    # Analyze citation pattern
    import re

    citation_numbers = re.findall(r"\[(\d+)\]", answer)
    print("\n📊 Citation Analysis:")
    print(f"  Total references: {len(refs)}")
    print(f"  Citations in text: {len(citation_numbers)}")
    print(f"  Unique citations: {len(set(citation_numbers))}")
    print(
        f"  Citation density: {len(citation_numbers) / len(answer.split()) * 100:.1f} citations per 100 words"
    )
else:
    print(f"Error: {response_6.get('error')}")

## 7. System Evaluation {#evaluation}

Run evaluation tasks to assess system performance.

In [ ]:
# Start an evaluation task
eval_request = {
    "task": "span",
    "dataset": "gold_span.jsonl",
    "model": "harvester/miner/ner_model/model-best",
    "metrics": ["precision", "recall", "f1"],
    "config": {
        "match_threshold": 0.5,
        "entity_types": ["MATERIAL", "TEMP", "TIME", "EQUIPMENT"],
    },
}

response_7 = make_request("/evaluate", "POST", eval_request)

if response_7.get("ok"):
    eval_id = response_7["evaluation_id"]
    print(f"🔬 Evaluation started: {eval_id}")
    print(f"Task: {response_7['task']}")
    print(f"Estimated time: {response_7['estimated_time']}")

    # Wait for completion (simplified for demo)
    time.sleep(5)
    results_response = make_request(f"/evaluate/{eval_id}")

    if results_response.get("ok") and results_response["status"] == "completed":
        results = results_response["results"]

        print("\n📊 Evaluation Results:")
        print("=" * 50)

        # Overall metrics
        overall = results["overall"]
        print("Overall Performance:")
        print(f"  Precision: {overall['precision']:.3f}")
        print(f"  Recall: {overall['recall']:.3f}")
        print(f"  F1 Score: {overall['f1']:.3f}")

        # Per-entity metrics
        print("\nPer-Entity Performance:")
        by_entity = results["by_entity"]
        for entity, metrics in by_entity.items():
            print(f"  {entity}:")
            print(f"    Precision: {metrics['precision']:.3f}")
            print(f"    Recall: {metrics['recall']:.3f}")
            print(f"    F1: {metrics['f1']:.3f}")

        # Visualize results
        entities = list(by_entity.keys())
        f1_scores = [by_entity[entity]["f1"] for entity in entities]

        plt.figure(figsize=(10, 6))
        bars = plt.bar(entities, f1_scores, alpha=0.7)
        plt.title("F1 Scores by Entity Type")
        plt.xlabel("Entity Type")
        plt.ylabel("F1 Score")
        plt.ylim(0, 1)

        # Add value labels on bars
        for bar, score in zip(bars, f1_scores):
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{score:.3f}",
                ha="center",
                va="bottom",
            )

        plt.tight_layout()
        plt.show()

    else:
        print(
            f"Evaluation not completed yet or failed: {results_response.get('status')}"
        )
else:
    print(f"Error starting evaluation: {response_7.get('error')}")

## 8. Advanced Features {#advanced}

Demonstrate advanced features like verbatim mode and knowledge base search.

In [ ]:
# Test verbatim mode (requires uploaded document)
verbatim_query = {
    "question": "Please quote the experimental procedure verbatim from the uploaded protocol",
    "mode": "protocol",
}

# Create a sample document for verbatim extraction
verbatim_doc = """
EXACT EXPERIMENTAL PROCEDURE

1. Dissolve 0.1 g HAuCl₄·3H₂O in 100 mL distilled water
2. Heat solution to exactly 100°C ± 2°C
3. Add 10 mL of 1% sodium citrate solution dropwise
4. Maintain temperature for precisely 15 minutes
5. Cool rapidly in ice bath to 25°C
6. Centrifuge at 10,000 rpm for 20 minutes
"""

verbatim_file = Path("verbatim_protocol.txt")
verbatim_file.write_text(verbatim_doc)

# Upload and query with verbatim request
with open(verbatim_file, "rb") as f:
    files = {"file": ("verbatim_protocol.txt", f, "text/plain")}
    data = {
        "question": "Quote the experimental procedure exactly as written",
        "mode": "protocol",
    }

    response_8 = make_request("/ask", "POST", data, files)

if response_8.get("ok"):
    print("📄 Verbatim Extraction:")
    print("=" * 50)
    print(response_8["answer"])

    if "rationale" in response_8:
        print(f"\n🔍 Extraction Method: {response_8['rationale']}")
else:
    print(f"Error: {response_8.get('error')}")

verbatim_file.unlink()

In [ ]:
# Direct knowledge base search
kb_search_params = {"q": "quantum dots photoluminescence", "k": 5, "threshold": 0.7}

response_9 = make_request("/kb/search", "GET", kb_search_params)

if response_9.get("ok"):
    print("🔍 Knowledge Base Search Results:")
    print("=" * 50)

    results = response_9["results"]
    for i, result in enumerate(results, 1):
        print(f"Result {i} (Score: {result['score']:.3f}):")
        print(f"  Title: {result.get('title', 'N/A')}")
        print(f"  Text: {result['text'][:200]}...")
        print(f"  Source: {result.get('source', 'N/A')}")
        print()

    print(f"Total results found: {response_9['total_results']}")
else:
    print(f"Error: {response_9.get('error')}")

## 9. Performance Analysis

Analyze system performance and response characteristics.

In [ ]:
# Performance benchmarking
test_queries = [
    "How to synthesize gold nanoparticles?",
    "What is the mechanism of silver nanoparticle formation?",
    "Optimize conditions for quantum dot synthesis",
    "Compare different reducing agents for metal nanoparticles",
    "What are the safety considerations for nanomaterial synthesis?",
]

performance_data = []

for i, query in enumerate(test_queries):
    print(f"Testing query {i+1}/5: {query[:50]}...")

    start_time = time.time()
    response = make_request("/ask", "POST", {"question": query, "mode": "protocol"})
    end_time = time.time()

    if response.get("ok"):
        response_time = end_time - start_time
        answer_length = len(response["answer"])
        num_refs = len(response.get("refs", []))

        performance_data.append(
            {
                "query": query,
                "response_time": response_time,
                "answer_length": answer_length,
                "num_references": num_refs,
                "words_per_second": len(response["answer"].split()) / response_time,
            }
        )
    else:
        print(f"  Failed: {response.get('error')}")

# Create performance summary
if performance_data:
    df = pd.DataFrame(performance_data)

    print("\n📊 Performance Summary:")
    print("=" * 50)
    print(f"Average response time: {df['response_time'].mean():.2f} seconds")
    print(f"Average answer length: {df['answer_length'].mean():.0f} characters")
    print(f"Average references: {df['num_references'].mean():.1f}")
    print(f"Average generation speed: {df['words_per_second'].mean():.1f} words/second")

    # Visualize performance
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 8))

    # Response times
    ax1.bar(range(len(df)), df["response_time"])
    ax1.set_title("Response Times")
    ax1.set_xlabel("Query Number")
    ax1.set_ylabel("Time (seconds)")

    # Answer lengths
    ax2.bar(range(len(df)), df["answer_length"])
    ax2.set_title("Answer Lengths")
    ax2.set_xlabel("Query Number")
    ax2.set_ylabel("Characters")

    # Number of references
    ax3.bar(range(len(df)), df["num_references"])
    ax3.set_title("Number of References")
    ax3.set_xlabel("Query Number")
    ax3.set_ylabel("References")

    # Generation speed
    ax4.bar(range(len(df)), df["words_per_second"])
    ax4.set_title("Generation Speed")
    ax4.set_xlabel("Query Number")
    ax4.set_ylabel("Words/Second")

    plt.tight_layout()
    plt.show()

    # Display detailed results
    print("\n📋 Detailed Results:")
    print(
        df[
            ["response_time", "answer_length", "num_references", "words_per_second"]
        ].round(2)
    )
else:
    print("No performance data collected")

## 10. System Information

Get comprehensive system information and capabilities.

In [ ]:
# Get system information
response_10 = make_request("/info")

if response_10.get("ok"):
    info = response_10

    print("🖥️ System Information:")
    print("=" * 50)
    print(f"Version: {info.get('version', 'Unknown')}")

    print("\n🔧 Capabilities:")
    capabilities = info.get("capabilities", {})
    for feature, enabled in capabilities.items():
        status = "✅" if enabled else "❌"
        print(f"  {status} {feature.replace('_', ' ').title()}")

    print("\n🤖 Models:")
    models = info.get("models", {})
    for model_type, model_name in models.items():
        print(f"  {model_type.title()}: {model_name}")

    print("\n📚 Index Statistics:")
    index_stats = info.get("index_stats", {})
    for stat, value in index_stats.items():
        print(f"  {stat.replace('_', ' ').title()}: {value}")

    print("\n💻 System Resources:")
    resources = info.get("system_resources", {})
    for resource, usage in resources.items():
        print(f"  {resource.replace('_', ' ').title()}: {usage}")
else:
    print(f"Error getting system info: {response_10.get('error')}")

## Conclusion

This notebook has demonstrated the comprehensive capabilities of NanoChemGPT:

1. **Question Answering**: Context-aware responses with proper citations
2. **File Processing**: Upload and analysis of synthesis protocols
3. **Protocol Conversion**: Translation to robot-executable operations
4. **Literature Mining**: Automated harvesting and indexing
5. **Citation Management**: Proper academic reference formatting
6. **System Evaluation**: Performance metrics and quality assessment
7. **Advanced Features**: Verbatim extraction and knowledge base search

### Next Steps

- Explore domain-specific queries in your area of interest
- Upload your own synthesis protocols for analysis
- Integrate with laboratory automation systems
- Contribute to the knowledge base with novel protocols
- Develop custom evaluation metrics for your use case

### Resources

- [API Documentation](../docs/API.md)
- [Installation Guide](../docs/INSTALLATION.md)
- [GitHub Repository](https://github.com/DMCarnahan/NanoChemGPT)
- [Issue Tracker](https://github.com/DMCarnahan/NanoChemGPT/issues)